In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


mortality = pd.read_excel("data/UNIGME-2025-UNICEFRegion-Rates-Deaths-Sex-specific-5-to-24.xlsx",sheet_name="20q5, 10q10",skiprows=14)

mortality.columns = mortality.columns.str.strip()

mortality = mortality[["Region", "Sex", "Year", "Median"]] #data analizi için only relevant variablesları seçtim

mortality = mortality.rename(columns={
    "Median": "mort_median_5_24"
})

mortality = mortality.groupby(["Region", "Year"])["mort_median_5_24"].mean().reset_index()
# veriyi region + year bazında ortalama alarak özetliyoruz, ayrıca bu adımla sex ve region level kısmını datasetten çıkardık
hdi = pd.read_excel(
    "data/HDR25_Statistical_Annex_HDI_Table.xlsx",
    skiprows=4,
    usecols="B:C"
)




mapping = pd.read_excel("data/JME_Regional-Classifications.xlsx")

mapping = mapping[["Country", "UNICEF Region"]]
#mapping = mapping.rename(columns={"UNICEF Region": " UNICEF Region"})
mapping.head()
econ = pd.read_excel(
    "data/19-Economic-Indicators-SOWC2025-1.xlsx",
    skiprows=6,
    header=None,
    usecols=[1, 6, 8]
)

econ.columns = ["Country", "health_spending", "education_spending"]

econ = econ.iloc[1:].reset_index(drop=True)

econ["health_spending"] = pd.to_numeric(econ["health_spending"], errors="coerce")
econ["education_spending"] = pd.to_numeric(econ["education_spending"], errors="coerce")

econ = econ.merge(mapping, on="Country", how="left")#burada econ datasındaki ülkelere, mapping datasından hangi bölgeye ait olduklarını ekledik
econ.head()
hdi.columns = ["Country", "HDI"]
hdi = hdi.iloc[2:].reset_index(drop=True)#gereksiz satırları silip index’i resetledim
hdi.head()
hdi = hdi.merge(mapping, on="Country", how="left")#burada da HDI verisindeki ülkelere hangi bölgeye ait olduklarını ekledik
hdi.head()
hdi = hdi.rename(columns={"UNICEF Region": "Region"})
hdi["HDI"] = pd.to_numeric(hdi["HDI"], errors="coerce")

hdi_region = hdi.groupby("Region")["HDI"].mean().reset_index()#ülkeleri bölgelere göre gruplayıp ortalama HDI hesaplıyoruz
hdi_region.head(10)


region_map = {
    "EAPRO": "East Asia and Pacific",
    "ECA": "Europe and Central Asia",
    "LACRO": "Latin America and Caribbean",
    "MENA": "Middle East and North Africa",
    "ROSA": "South Asia",
    "SSA": "Sub-Saharan Africa"
}
#burada kısaltılmış region kodlarını tam isimlere çevirmek için mapping oluşturdum(gpt söyledi)
hdi_region["Region"] = hdi_region["Region"].replace(region_map)
hdi_region

econ_region = econ.groupby("UNICEF Region")[["health_spending", "education_spending"]].mean().reset_index()
econ_region["UNICEF Region"] = econ_region["UNICEF Region"].replace(region_map)
econ_region = econ_region.rename(columns={"UNICEF Region": "Region"}) #Ülkeleri bölgelere göre gruplayıp: health spending ortalaması ve education spending ortalaması aldık
#(groupby ile)

df = mortality.merge(hdi_region, on="Region", how="left") #burada mortality datasına hdi ekledim
df = df.merge(econ_region, on="Region", how="left")#burada da economic verileri ekledim

df.head()
df.isnull().sum()#merge sonrası verileri kontrol eder her sütun biribriyle eşleniyor mu boş veri var mı diye

fix_map = {
    "Eastern Europe and Central Asia": "Europe and Central Asia",
    "Western Europe": "Europe and Central Asia",
    "Eastern and Southern Africa": "Sub-Saharan Africa",
    "West and Central Africa": "Sub-Saharan Africa",
    "North America": "Europe and Central Asia",  # approx (dataset'e göre kabul)
    "World": None  # bunu istemiyoruz
}

df["Region"] = df["Region"].replace(fix_map)#mortality datasetindeki region isimlerini diğer datasetlerle uyumlu hale getirdik
df = mortality.copy()
df["Region"] = df["Region"].replace(fix_map)

df = df.merge(hdi_region, on="Region", how="left")
df = df.merge(econ_region, on="Region", how="left")
df.isnull().sum()
df = df[df["Region"].notna()].reset_index(drop=True)# burada region değeri boş (NaN) olan satırları sildik, sonra index’i sıfırlayıp düzenledik
df.isnull().sum()






Region                0
Year                  0
mort_median_5_24      0
HDI                   0
health_spending       0
education_spending    0
dtype: int64

###HYPOTHESIS 1

This analysis examines whether health spending or education spending has a stronger effect on youth mortality rates.


*Hypotheses*

**H0 (Null Hypothesis):**
Health and education spending have similar relationships with youth mortality rates.

**H1 (Alternative Hypothesis):**
One of these variables has a stronger relationship with youth mortality rates.

In [2]:
df[["health_spending", "education_spending", "mort_median_5_24"]].corr()

,health_spending,education_spending,mort_median_5_24
health_spending,1.000000,0.987001,-0.771455
education_spending,0.987001,1.000000,-0.741686
mort_median_5_24,-0.771455,-0.741686,1.000000


here you can see from the correlation matrix shows that both health and education spending are negatively associated with youth mortality rates. However, education spending appears to have a stronger relationship with mortality reduction compared to health spending.
also, these relationships are moderate and do not imply causation

In [3]:
from scipy.stats import pearsonr

corr, p_value = pearsonr(df["HDI"], df["mort_median_5_24"])
print(corr, p_value)

-0.8305742900201275 2.11441803756972e-99


above we calculated the Pearson correlation coefficient and its p-value to test the strength and statistical significance of the relationship between HDI and youth mortality.
note:	•ilişki gücünü (corr)
	    •bu ilişkinin anlamlı olup olmadığını (p-value) verir

note: the relationship is statistically significant (p < 0.001).
note:p-value (correlation testi) == “bu gerçek mi?” testi 
and 
above, we use "scipy.stats" to make statistical calculations
note:  ANOVA = “gruplar gerçekten farklı mı?” testi

In [4]:
from scipy.stats import f_oneway

groups = [group["mort_median_5_24"].values for name, group in df.groupby("Region")]

f_stat, p_value = f_oneway(*groups)

print(f_stat, p_value)

276.25730462245025 5.677159046068467e-124


we performed a one-way ANOVA test to determine whether there are statistically significant differences in youth mortality rates across regions. 
moreover, the one-way ANOVA test shows that there are statistically significant differences in youth mortality rates across regions (p < 0.05). This indicates that mortality levels vary meaningfully between different parts of the world.


###HYPOTHESIS 2

When development increases, mortality goes down. The variation in mortality may also become smaller.

⸻

**Hypotheses**

*H0 (Null Hypothesis):*
There is no difference in mortality variation between high-HDI and low-HDI groups.

*H1 (Alternative Hypothesis):*
High-HDI groups have less variation in mortality than low-HDI groups.

In [5]:
high = df[df["HDI"] >= df["HDI"].median()]["mort_median_5_24"]
low = df[df["HDI"] < df["HDI"].median()]["mort_median_5_24"]

high.var(), low.var()

(23.444783186404887, 513.7750122562574)

Development reduces not only mortality but also uncertainty.

###Hypothesis3


**H0:**
After controlling for HDI, regions do not differ in youth mortality

**H1:**
Even after controlling for HDI, regions still differ

In [6]:
import statsmodels.api as sm

X = df[["HDI"]]
y = df["mort_median_5_24"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       mort_median_5_24   R-squared:                       0.690
Model:                            OLS   Adj. R-squared:                  0.689
Method:                 Least Squares   F-statistic:                     851.9
Date:                Fri, 15 May 2026   Prob (F-statistic):           2.11e-99
Time:                        13:11:47   Log-Likelihood:                -1494.8
No. Observations:                 385   AIC:                             2994.
Df Residuals:                     383   BIC:                             3002.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        124.9671      3.477     35.938      0.0

interpretation for this: HDI has a strong and statistically significant negative effect on youth mortality (β = -134.82, p < 0.001). This suggests that higher development levels are associated with substantially lower mortality rates. The model explains a large portion of the variation in mortality (R² = 0.69), indicating that HDI alone is a powerful predictor.

In [7]:
import statsmodels.formula.api as smf

model = smf.ols("mort_median_5_24 ~ HDI + C(Region)", data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       mort_median_5_24   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.782
Method:                 Least Squares   F-statistic:                     276.3
Date:                Fri, 15 May 2026   Prob (F-statistic):          5.68e-124
Time:                        13:11:47   Log-Likelihood:                -1424.6
No. Observations:                 385   AIC:                             2861.
Df Residuals:                     379   BIC:                             2885.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

A regression model including both HDI and region indicates that some regional differences remain significant even after controlling for development. In particular, Sub-Saharan Africa and South Asia show significantly higher mortality levels. However, the HDI coefficient is not statistically significant in this model, likely due to strong multicollinearity between HDI and regional structure.

 ###Conclusion

We cannot fully accept the null hypothesis, because some regional variables are still significant.
This means that even after controlling for HDI, some regions still have different mortality rates.
⸻
**Important Note**
The model shows strong multicollinearity.
This means:
	*	HDI and Region are closely related
	*	HDI may look insignificant in the model
	*	but this does not mean HDI is unimportant
	*	it means it is hard to separate the effect of HDI from region
⸻
Therefore, although HDI is a strong predictor of youth mortality, its effect becomes weaker when regional variables are included, suggesting that development is closely linked to regional conditions.

In [8]:
import statsmodels.formula.api as smf

model = smf.ols(
    "mort_median_5_24 ~ HDI + health_spending + education_spending + C(Region)",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       mort_median_5_24   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.782
Method:                 Least Squares   F-statistic:                     276.3
Date:                Fri, 15 May 2026   Prob (F-statistic):          5.68e-124
Time:                        13:11:47   Log-Likelihood:                -1424.6
No. Observations:                 385   AIC:                             2861.
Df Residuals:                     379   BIC:                             2885.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

In [9]:
import statsmodels.formula.api as smf

model = smf.ols(
    "mort_median_5_24 ~ HDI + health_spending + education_spending + C(Region)",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       mort_median_5_24   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.782
Method:                 Least Squares   F-statistic:                     276.3
Date:                Fri, 15 May 2026   Prob (F-statistic):          5.68e-124
Time:                        13:11:47   Log-Likelihood:                -1424.6
No. Observations:                 385   AIC:                             2861.
Df Residuals:                     379   BIC:                             2885.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------


## Regression Summary (HDI + Spending + Region)

 **Model Performance**
* R² ≈ 0.78 → strong explanatory power
* Model is statistically significant (p < 0.001)

👌🏼 The model explains most of the variation in youth mortality.

⸻
 **Health Spending**

* Negative and significant effect (p < 0.001)

👌🏼 Higher health spending is associated with lower youth mortality
👌🏼 Most reliable and clear predictor

⸻
** Education Spending**

* Positive and significant coefficient (p < 0.001)

🪻 Interpretation should be cautious
👌🏼 Likely affected by multicollinearity

👌🏼 Does NOT mean education increases mortality
👌🏼 Indicates overlapping effects with other variables

⸻

** HDI**

* Not statistically significant (p > 0.05)

👌🏼 When more specific variables are included,
HDI’s independent effect becomes unclear

👌🏼 Suggests HDI overlaps with spending variables

⸻

**Regional Effects**

* Several regions remain statistically significant

👌🏼 Even after controlling for HDI and spending:
mortality differs across regions

👌🏼 Indicates presence of structural / geographic factors

⸻

* Multicollinearity*

* Strong correlation between HDI, health, and education

👌🏼 Makes it difficult to isolate individual effects
👌🏼 Coefficients should be interpreted carefully

⸻

*Final Insight*

👌🏼 Youth mortality is not explained by development alone

👌🏼 It is shaped by:

* targeted investments (especially health)
* regional structural differences

